# llm-d Serving with Valkey: Routing, KV Cache and Batch Queue

> Put llm-d's router in front of two CPU vLLM servers and use one Valkey for the router's prefix-cache index, the shared KV cache, and the batch job queue with per-user quotas.

**Difficulty:** Advanced · Python · ~30 min

**Who is this for:** Engineers serving LLMs on more than one vLLM instance who want to see what a cache-aware router does, and what Valkey stores for it, before they deploy one.

This recipe builds on the [KV caching recipe](../kv-caching/), where two vLLM
servers share one KV cache in Valkey through LMCache. Sharing a cache means a
server can load a prompt prefix that another server computed instead of
recomputing it. It is still slower than finding the prefix in the server's own
memory, and with two or more servers somebody has to decide which server gets
each request.

That is the job of [llm-d](https://llm-d.ai)'s router: Envoy plus llm-d's
Endpoint Picker (EPP). The router listens to each vLLM server's KV-cache events,
keeps an index of which server holds which prompt blocks, and sends each request
to the server that already has the longest matching prefix, unless that server
is too busy. Here it runs without Kubernetes, from a static endpoint file.

In this stack one Valkey has three jobs:

1. **Routing index.** The router keeps its block-to-server index in Valkey, so
   the index lives outside the router process: a restarted router reads the
   same index.
2. **KV cache.** When the router sends a conversation to a server that does not
   have it in memory, for example because the best server is busy, that server
   loads the conversation's KV from Valkey through LMCache instead of
   recomputing it.
3. **Batch queue and quotas.** llm-d's batch processor (llm-d-async) keeps its
   job queue and per-user rate limits in Valkey and sends admitted jobs through
   the same router.

None of the llm-d components is modified: each is the upstream image, pointed at
Valkey by configuration.

llm-d is an open-source framework for serving LLMs across several vLLM servers.
Its router sends each request to a server based on what it already has cached
and how busy it is.

## How to run this notebook

The [recipe README](README.md) has the prerequisites (a container runtime with
about 10 GiB of memory, and [uv](https://docs.astral.sh/uv/)) and the one-time
setup. In short, from this directory:

```bash
docker compose up -d --wait
uv sync
uv run python -m ipykernel install --user --name llm-d-serving-valkey
uv run jupyter lab llm-d-serving-with-valkey.ipynb
```

> ⚠ **Security:** These examples use no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

Select the `llm-d-serving-valkey` kernel and run the cells top to bottom. Later
cells build on earlier ones. A few cells restart or reconfigure the router with
`docker compose` (or `podman compose`; the notebook uses whichever is
installed), so run the notebook from this directory. A full run takes two to three
minutes on an 8-core laptop.

## Connect to the stack

The setup cell first checks that the container runtime has the 10 GiB of memory
this stack needs, and stops with a message if it doesn't. Then it defines the
clients and a few helpers that every later cell uses:

- `router` sends requests through Envoy on `localhost:8080`, so the EPP picks
  the replica. `direct["vllm-a"]` and `direct["vllm-b"]` skip the router.
- `valkey` reads database 0, where LMCache stores KV chunks and llm-d-async
  keeps its queue. `router_index` reads database 1, where the router keeps its
  index. (Why the index has a database of its own is explained in
  "What the router stores in Valkey".)
- `ask()` sends one streaming chat request and reports the time to the first
  token, how many prompt tokens were cached (`cached_tokens` in the response),
  and which replica served it. vLLM's responses do not name the replica, so
  `ask()` reads each replica's `vllm:request_success_total` counter before and
  after and sees which one went up. It also reports where the cached tokens came
  from: `vllm:prefix_cache_hits_total` counts tokens found in the replica's own
  memory and `vllm:external_prefix_cache_hits_total` counts tokens LMCache
  loaded from Valkey.
- `settled_dbsize()` waits until database 0 stops growing. The LMCache server
  writes KV to Valkey asynchronously, so before a cell expects another replica
  to load a prefix from Valkey, it waits for those writes to land.

In [ ]:
import http.client
import json
import os
import re
import secrets
import shutil
import socket
import statistics
import subprocess
import time
import urllib.request
import uuid

from glide_sync import GlideClient, GlideClientConfiguration, NodeAddress, Script
import yaml
from openai import OpenAI

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ROUTER_URL = "http://127.0.0.1:8080"  # Envoy; the EPP picks the replica
REPLICA_URLS = {"vllm-a": "http://127.0.0.1:8001", "vllm-b": "http://127.0.0.1:8002"}
# The router names replicas by "<ip>:<port>"; epp/endpoints.yaml maps those to names.
with open("epp/endpoints.yaml") as f:
    POD_NAMES = {f"{e['address']}:{e['port']}": e["name"] for e in yaml.safe_load(f)["endpoints"]}
EPP_METRICS = "http://127.0.0.1:9090/metrics"
ASYNC_METRICS = "http://127.0.0.1:9091/metrics"
VALKEY_HOST = os.environ.get("VALKEY_HOST", "127.0.0.1")
VALKEY_PORT = int(os.environ.get("VALKEY_PORT", "6379"))

router = OpenAI(base_url=f"{ROUTER_URL}/v1", api_key="not-needed", max_retries=0)
direct = {name: OpenAI(base_url=f"{url}/v1", api_key="not-needed", max_retries=0)
          for name, url in REPLICA_URLS.items()}


def connect(database_id):
    return GlideClient.create(GlideClientConfiguration(
        addresses=[NodeAddress(VALKEY_HOST, VALKEY_PORT)],
        database_id=database_id, request_timeout=5000))


valkey = connect(0)        # LMCache KV chunks + llm-d-async queue and quotas
router_index = connect(1)  # the router's prefix-cache index


def settled_dbsize(quiet_s=1.0, timeout_s=15.0):
    """Database 0's DBSIZE once it has not changed for `quiet_s` seconds."""
    deadline = time.monotonic() + timeout_s
    last, since = valkey.custom_command(["DBSIZE"]), time.monotonic()
    while time.monotonic() < deadline:
        time.sleep(0.1)
        current = valkey.custom_command(["DBSIZE"])
        if current != last:
            last, since = current, time.monotonic()
        elif time.monotonic() - since >= quiet_s:
            break
    return last


def detect_engine():
    """First of docker/podman that is installed and answers `info`."""
    forced = os.environ.get("CONTAINER_ENGINE")
    for candidate in [forced] if forced else ["docker", "podman"]:
        if candidate and shutil.which(candidate):
            if subprocess.run([candidate, "info"], capture_output=True, timeout=30).returncode == 0:
                return candidate
    raise RuntimeError("No working container runtime found (tried docker, podman).")


ENGINE = detect_engine()

# Preflight: stop now if the runtime has less memory than the stack needs,
# instead of a replica dying halfway through the notebook. A runtime given
# 10 GiB reports a little less (the VM keeps some), so the cutoff is 9.5 GiB,
# the same as the README's startup check.
MIN_RUNTIME_MEMORY = 10_200_547_328


def runtime_memory_bytes():
    """Memory the container runtime reports (Docker, then Podman's field), or None."""
    for fmt in ("{{.MemTotal}}", "{{.Host.MemTotal}}"):
        out = subprocess.run([ENGINE, "info", "--format", fmt], capture_output=True,
                             text=True, timeout=30).stdout.strip()
        if out.isdigit():
            return int(out)
    return None


RUNTIME_MEMORY = runtime_memory_bytes()
if RUNTIME_MEMORY is not None and RUNTIME_MEMORY < MIN_RUNTIME_MEMORY:
    raise SystemExit(
        f"{ENGINE} reports {RUNTIME_MEMORY / 2**30:.1f} GiB of memory; this recipe needs 10 GiB. "
        "Raise the runtime's memory limit (README, 'Memory requirements'), restart the stack, "
        "and run this cell again.")


def compose(*args, **env):
    """Run `<engine> compose ...` in this directory, with extra env vars."""
    result = subprocess.run([ENGINE, "compose", *args], capture_output=True, text=True,
                            env={**os.environ, **env}, timeout=600)
    if result.returncode != 0:
        raise RuntimeError(f"{ENGINE} compose {' '.join(args)} failed:\n{result.stderr}")


def scrape(url):
    with urllib.request.urlopen(url, timeout=5) as resp:
        return resp.read().decode()


def metric(text, name, **labels):
    """Sum every sample of a Prometheus metric whose labels include `labels`."""
    total = 0.0
    for match in re.finditer(rf"^{re.escape(name)}(?:\{{([^}}]*)\}})? (\S+)$", text, re.M):
        found = dict(re.findall(r'(\w+)="([^"]*)"', match.group(1) or ""))
        if all(found.get(k) == v for k, v in labels.items()):
            total += float(match.group(2))
    return total


def replica_counters():
    counters = {}
    for name, url in REPLICA_URLS.items():
        text = scrape(f"{url}/metrics")
        counters[name] = {
            "served": metric(text, "vllm:request_success_total"),
            "local": metric(text, "vllm:prefix_cache_hits_total"),
            "valkey": metric(text, "vllm:external_prefix_cache_hits_total"),
        }
    return counters


def served_by(before, timeout_s=3.0):
    """Which replica finished a request since `before`, plus its cache-hit deltas."""
    deadline = time.monotonic() + timeout_s
    while True:
        after = replica_counters()
        moved = [n for n in after if after[n]["served"] > before[n]["served"]]
        if moved or time.monotonic() > deadline:
            break
        time.sleep(0.05)
    name = moved[0] if len(moved) == 1 else "?"
    if name == "?":
        return name, 0, 0
    return (name, int(after[name]["local"] - before[name]["local"]),
            int(after[name]["valkey"] - before[name]["valkey"]))


def ask(client, messages, max_tokens=16):
    """One streaming chat request; returns timing, cache and placement signals."""
    before = replica_counters()
    start = time.perf_counter()
    ttft, text, usage = None, "", None
    stream = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0,
        stream=True, stream_options={"include_usage": True})
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            ttft = ttft or (time.perf_counter() - start)
            text += chunk.choices[0].delta.content
        if chunk.usage:
            usage = chunk.usage
    replica, local, from_valkey = served_by(before)
    details = usage.prompt_tokens_details
    return {"replica": replica, "ttft_ms": round((ttft or time.perf_counter() - start) * 1000),
            "prompt": usage.prompt_tokens, "cached": details.cached_tokens if details else 0,
            "local": local, "valkey": from_valkey, "text": text}


memory = f"{RUNTIME_MEMORY / 2**30:.1f} GiB" if RUNTIME_MEMORY else "not reported"
print(f"container runtime: {ENGINE}, memory {memory}")
print("Setup ready.")

Check that everything is up before going further: Valkey answers, both replicas
are healthy, and a request through the router succeeds. The cell also makes sure
the router runs its default configuration (`epp/precise-valkey.yaml`), in case
an earlier run stopped halfway through the random-routing section. Envoy and the
EPP images have no shell for a container healthcheck, so `docker compose up
--wait` cannot tell when the router is ready; `wait_for_router()` retries a
one-token request until one goes through.

In [ ]:
def wait_for_router(timeout_s=120):
    deadline = time.monotonic() + timeout_s
    while True:
        try:
            router.completions.create(model=MODEL, prompt="ping", max_tokens=1, timeout=10)
            return
        except Exception as err:  # connection refused / 503 while the EPP starts
            if time.monotonic() > deadline:
                raise RuntimeError("The router did not answer; check `docker compose logs epp`.") from err
            time.sleep(1)


def use_routing(strategy):
    """Recreate the EPP with epp/<strategy>.yaml (no-op if it already runs it)."""
    compose("up", "-d", "epp", EPP_STRATEGY=strategy)
    wait_for_router()


assert valkey.ping() in (b"PONG", "PONG"), "Valkey is not answering"
for name, url in REPLICA_URLS.items():
    with urllib.request.urlopen(f"{url}/health", timeout=5) as resp:
        assert resp.status == 200, f"{name} is not healthy"
use_routing("precise-valkey")
print("valkey      : PONG")
print("replicas    :", ", ".join(REPLICA_URLS))
print("router      : answering on", ROUTER_URL, "(precise-valkey)")

## What Valkey holds

Three kinds of keys will show up in Valkey as the notebook runs:

| Database | Keys | Written by | What it is |
| --- | --- | --- | --- |
| 1 | `engine:<hash>` (sorted set) | router | maps a vLLM block hash to the router's own hash for the same block |
| 1 | `<hash>` (hash) | router | one key per cached 64-token block; each field names a replica that holds it |
| 0 | `Qwen/Qwen2.5-0.5B-Instruct@...` (string) | LMCache server | one 256-token KV chunk, about 3 MiB |
| 0 | `request-sortedset`, `result-list`, `quota:*`, `request-*` | llm-d-async | batch job queue, results, per-user quotas, per-job state |

`key_census()` counts them. Right after startup the router index already has a
few entries: each replica's healthcheck warm-up filled one 64-token block. There
are no LMCache chunks yet, because the warm-up is shorter than one 256-token
chunk. (On a re-run you will also see what earlier runs left behind; nothing in
the notebook depends on an empty Valkey.)

In [ ]:
QUEUE_KEYS = ("request-sortedset", "result-list")
QUEUE_PREFIXES = ("quota:", "request-seq:", "request-active:", "request-cancel:", "request-payload:")


def scan_keys(client):
    cursor, keys = b"0", []
    while True:
        cursor, batch = client.scan(cursor, count=1000)
        keys += [k.decode() for k in batch]
        if cursor in (b"0", "0"):
            return keys


def key_census():
    census = {"router engine:<hash>": 0, "router <hash>": 0, "LMCache chunks": 0,
              "batch queue": 0, "other": 0}
    for key in scan_keys(router_index):
        census["router engine:<hash>" if key.startswith("engine:") else
               "router <hash>" if key.isdigit() else "other"] += 1
    for key in scan_keys(valkey):
        census["LMCache chunks" if key.startswith(f"{MODEL}@") else
               "batch queue" if key in QUEUE_KEYS or key.startswith(QUEUE_PREFIXES) else
               "other"] += 1
    return census


def show_census(title):
    print(f"{title}: DBSIZE db0={valkey.custom_command(['DBSIZE'])} "
          f"db1={router_index.custom_command(['DBSIZE'])}")
    for kind, count in key_census().items():
        print(f"  {kind:22s} {count:5d}")


show_census("Valkey now")

## Three ways to get the first token

How quickly a replica can start answering depends on what it already has for
the prompt. Where the router sends a request decides which of three cases it
lands in, so this section measures all three directly, without the router. The
prompt here is a 29-clause document of roughly 950 to 1,100 tokens including
the system instruction, plus a question. Its length varies with the random
tag repeated in each clause:

- **Recompute**: no replica has seen it. vLLM prefills every token.
- **Load from Valkey**: another replica computed it, so the KV is in Valkey but
  not in this replica's memory. LMCache loads it from Valkey in 256-token
  chunks and vLLM prefills only the rest.
- **Local hit**: this replica has it in its own memory (vLLM's prefix cache).

The cell sends the same fresh prompt to replica A (recompute), then to replica B
twice (load from Valkey, then local hit), directly rather than through the
router, and repeats with three different documents. Before B's first request the
cell waits for A's KV to reach Valkey (`settled_dbsize()`). `max_tokens=1`, so
the timing is prefill plus one token. A fresh `uuid` in each document keeps the
first request cold on every run.

In [ ]:
QUESTIONS = ["Summarise clause 3.", "What does clause 9 require?", "And clause 14?"]


def document(tag, clauses=29):
    """At 29 clauses, roughly 900–1,100 tokens, depending on `tag`."""
    return f"[document {tag}] " + " ".join(
        f"Clause {i}: the tenant of unit {tag} shall keep fixture {i * 7 % 13} in good "
        f"repair and report any fault within {i % 5 + 1} days." for i in range(clauses))


def new_conversation(label):
    tag = f"{label}-{uuid.uuid4().hex[:8]}"
    return [{"role": "system", "content": "Answer questions about this lease.\n\n" + document(tag)}]


cases = {"recompute": [], "load from Valkey": [], "local hit": []}
for _ in range(3):
    prompt = new_conversation("latency") + [{"role": "user", "content": QUESTIONS[0]}]
    cases["recompute"].append(ask(direct["vllm-a"], prompt, max_tokens=1))
    settled_dbsize()  # LMCache stores A's KV in Valkey asynchronously
    cases["load from Valkey"].append(ask(direct["vllm-b"], prompt, max_tokens=1))
    cases["local hit"].append(ask(direct["vllm-b"], prompt, max_tokens=1))

print(f"{'case':18s} {'TTFT ms (3 runs)':22s} {'median':>7s} {'cached':>7s} {'local':>6s} {'Valkey':>7s}")
for case, runs in cases.items():
    last = runs[-1]
    print(f"{case:18s} {str([r['ttft_ms'] for r in runs]):22s} "
          f"{statistics.median(r['ttft_ms'] for r in runs):7.0f} {last['cached']:7d} "
          f"{last['local']:6d} {last['valkey']:7d}   (of {last['prompt']} prompt tokens)")

prefill_rate = statistics.median(r["prompt"] / (r["ttft_ms"] / 1000) for r in cases["recompute"])
print(f"\nmeasured prefill rate on this machine: {prefill_rate:,.0f} tokens/s "
      "(epp/precise-valkey.yaml assumes 1,000)")

The local hit skips nearly all of the prefill. Loading from Valkey is slower
than a local hit, because about 3 MiB per 256-token chunk travels from Valkey to
the LMCache server to vLLM, and LMCache only stores whole chunks, so the last
partial chunk is recomputed (the `Valkey` column stops at a multiple of 256). It
is still several times faster than recomputing. On an 8-core laptop expect
roughly 1.2 s to recompute, 0.1 to 0.15 s to load from Valkey, and about 0.1 s
for a local hit; your numbers depend on the CPU. The `cached` column is what vLLM
reports to the client in `usage.prompt_tokens_details.cached_tokens`; it counts
both kinds of hit.

The last line is the prefill rate the recompute case implies. The router's load
estimate (used in "When the best server is busy") divides queued tokens by a
configured rate, `peakPrefillThroughput`, set to 1,000 tokens/s in
`epp/precise-valkey.yaml`. llm-d's default is 15,928,
calibrated for Qwen 32B on two H100 80GB GPUs with vLLM 0.19 (see the
[affinity filter's README](https://github.com/llm-d/llm-d-router/blob/v1.0.0/pkg/epp/framework/plugins/scheduling/filter/prefixcacheaffinity/README.md#config) in
llm-d-router v1.0.0).

The router's aim is for most requests to be local hits: it sends each request to
the replica that already has the prompt in memory. When a request goes to the
other replica anyway, that replica loads the prompt from Valkey instead of
recomputing it.

## Precise routing: follow-ups find their server

Now go through the router with its default configuration
([`epp/precise-valkey.yaml`](epp/precise-valkey.yaml)). The pieces that matter:

```yaml
- type: precise-prefix-cache-producer
  parameters:
    tokenProcessorConfig:
      blockSizeTokens: 64              # = vLLM --block-size
    indexerConfig:
      kvBlockIndexConfig:
        redisConfig:
          address: redis://valkey:6379/1   # the index lives in Valkey, database 1
    kvEventsConfig:
      discoverPods: true               # subscribe to every replica's KV events
- type: prefix-cache-affinity-filter   # keep only replicas holding >= 80% of the prompt
- type: token-load-scorer              # among those, pick the least-loaded
```

For every request the router tokenizes the prompt (by asking vLLM's
`/v1/chat/completions/render` endpoint), hashes it into 64-token blocks the same
way vLLM does, and looks the block hashes up in Valkey. Each replica publishes a
*block stored* or *block removed* event whenever its prefix cache changes; the
router subscribes to both replicas and keeps the index in Valkey up to date.

Four conversations, three turns each. Turn 1 is cold everywhere, so the router
places it by load (both replicas are idle, so the choice is effectively random).
Turns 2 and 3 resend the whole conversation, and the router should send them to
the replica that served turn 1.

In [ ]:
def run_conversations(client, label, count, turns=3):
    """Interleave `count` conversations turn by turn; return one row per request."""
    conversations = [new_conversation(f"{label}{i}") for i in range(count)]
    rows = []
    for turn in range(turns):
        for i, messages in enumerate(conversations):
            messages.append({"role": "user", "content": QUESTIONS[turn]})
            result = ask(client, messages)
            messages.append({"role": "assistant", "content": result["text"]})
            rows.append({"conversation": i, "turn": turn + 1, **result})
    return conversations, rows


def print_rows(rows):
    print("conv turn  replica  TTFT ms  cached  local  Valkey")
    for r in rows:
        print(f"{r['conversation']:4d} {r['turn']:4d}   {r['replica']:7s} {r['ttft_ms']:8d} "
              f"{r['cached']:7d} {r['local']:6d} {r['valkey']:7d}")


def follow_up_summary(rows):
    first = {r["conversation"]: r["replica"] for r in rows if r["turn"] == 1}
    follow = [r for r in rows if r["turn"] > 1]
    same = [r for r in follow if r["replica"] == first[r["conversation"]]]
    moved = [r for r in follow if r["replica"] != first[r["conversation"]]]
    print(f"\nfollow-ups on the replica that served turn 1: {len(same)}/{len(follow)}")
    for label, group in (("stayed", same), ("moved", moved)):
        if group:
            print(f"  {label:6s}: median TTFT {statistics.median(r['ttft_ms'] for r in group):5.0f} ms, "
                  f"tokens from local memory {sum(r['local'] for r in group):6d}, "
                  f"from Valkey {sum(r['valkey'] for r in group):6d}")
    return same, moved


precise_conversations, precise_rows = run_conversations(router, "precise", count=4)
print_rows(precise_rows)
follow_up_summary(precise_rows);

### What the router stores in Valkey

The router's index is in database 1. Each 64-token block of every prompt a
replica has cached is one Valkey hash, keyed by the router's hash of the
block's tokens (and of every block before it, so a hash identifies a whole
prefix). Each field in that hash is a small JSON record naming a replica that
holds the block. Fields with `"Speculative":true` are the router's own guess:
with `speculativeIndexing: true` it records a routed request's blocks against
the chosen replica for 2 seconds, before the replica's events confirm them.
`engine:<hash>` sorted sets map vLLM's own block hashes, which
arrive in the events, to the router's hashes. The cell below shows one entry and
counts the confirmed (non-speculative) blocks the index assigns to each
replica.

The index has a database of its own because of how the router forgets a
replica. When a replica restarts with an empty cache, the router drops that
replica's entries by `SCAN`ning its whole database and reading every key that
does not start with `engine:` as a hash (`RedisIndex.Clear` in llm-d-router
v1.0.0). In a database shared with LMCache's string keys, that scan stops at the
first LMCache key with a `WRONGTYPE` error and leaves stale entries pointing at
the restarted replica. Selecting database 1 in the router's URL (`/1`) avoids it
with configuration only.

In [ ]:
def confirmed_entries(key):
    """The block's fields that a replica reported (not the router's 2 s guesses)."""
    return [e for e in map(json.loads, router_index.hkeys(key)) if not e.get("Speculative")]


def index_by_replica():
    """Confirmed blocks per replica."""
    counts = {name: 0 for name in POD_NAMES.values()}
    for key in scan_keys(router_index):
        if key.isdigit():
            for entry in confirmed_entries(key):
                # An address not in epp/endpoints.yaml (say, left over from
                # before a subnet change) is counted under its own name.
                name = POD_NAMES.get(entry["PodIdentifier"], entry["PodIdentifier"])
                counts[name] = counts.get(name, 0) + 1
    return counts


# Show a block both replicas reported, such as the healthcheck warm-up block.
block_key = max((k for k in scan_keys(router_index) if k.isdigit()),
                key=lambda k: len({e["PodIdentifier"] for e in confirmed_entries(k)}))
engine_key = next(k for k in scan_keys(router_index) if k.startswith("engine:"))
print(f"{block_key} ({router_index.type(block_key).decode()}):")
for entry in confirmed_entries(block_key):
    print("   field:", json.dumps(entry, separators=(",", ":")))
print(f"{engine_key} ({router_index.type(engine_key).decode()}):",
      router_index.custom_command(["ZRANGE", engine_key, "0", "-1"]))
lmcache_key = next(k for k in scan_keys(valkey) if k.startswith(f"{MODEL}@"))
print(f"\nLMCache chunk {lmcache_key[:60]}... ({valkey.type(lmcache_key).decode()}, "
      f"{valkey.strlen(lmcache_key):,} bytes)")
print("\nblocks the router index assigns to each replica:", index_by_replica())
show_census("\nValkey now")

The block printed first is one that both replicas have cached, so its hash has
two fields, one per replica. In each field, `PodIdentifier` is the replica's
address and port from `epp/endpoints.yaml`. `DeviceTier` says `gpu` even
though these replicas run on CPU, because vLLM labels everything in its own
prefix cache `gpu`, whatever hardware it runs on.

## Random routing: what happens without the index

For a baseline, switch the router to [`epp/random.yaml`](epp/random.yaml): the
same Envoy and EPP, but the EPP picks a replica at random and does not look at
the index. The cell recreates the EPP container with
`EPP_STRATEGY=random docker compose up -d epp`.

Five conversations, three turns each, so ten follow-ups. With two replicas, a
follow-up has a coin-flip chance of landing on the replica that does not have
the conversation in memory. Without a shared cache that replica would recompute
the whole conversation. Here it loads the conversation from Valkey, which the
`Valkey` column shows.

In [ ]:
use_routing("random")
random_conversations, random_rows = run_conversations(router, "random", count=5)
print_rows(random_rows)
stayed, moved = follow_up_summary(random_rows)
use_routing("precise-valkey")
print("\nrouter switched back to precise-valkey")

Follow-ups that stayed on their replica were local hits. A follow-up that moved
to a replica without the conversation loaded it from Valkey: slower than a local
hit, still far below a recompute (compare with the table in "Three ways to get
the first token"). A follow-up that moves to a replica that already has the
conversation from an earlier moved turn is local again. Your counts differ
from run to run because the picks are random. Typically three to six of
the ten follow-ups move; on an 8-core laptop the ones that stay take around 0.1 s and
the ones that move around 0.15 to 0.2 s, against about 1.2 s to recompute.

The router and the shared cache do different jobs. The router sends each
follow-up to the replica that already has the conversation in memory, so most
requests are local hits. When a request goes to the other replica anyway,
LMCache loads the conversation from Valkey instead of recomputing it.

## Restarting the router

The router keeps no index of its own: it is in Valkey. Start three new
conversations, restart the EPP container, and continue them. The router's block
keys in database 1 are the same after the restart, and the follow-ups go back to
the replica that has each conversation.

In [ ]:
def orphaned_engine_keys():
    """engine:<hash> mappings whose router block keys no longer exist."""
    orphans = 0
    for key in scan_keys(router_index):
        if key.startswith("engine:"):
            blocks = router_index.custom_command(["ZRANGE", key, "0", "-1"])
            orphans += not any(router_index.exists([b]) for b in blocks)
    return orphans


restart_conversations, before_rows = run_conversations(router, "restart", count=3, turns=1)
# The router drops its 2 s speculative entries on a timer in its own process; a
# restart inside that window would leave them in Valkey. Let them expire first.
time.sleep(2.5)
census_before, index_before = key_census(), index_by_replica()
orphans_before = orphaned_engine_keys()

compose("restart", "epp")
wait_for_router()

census_after, index_after = key_census(), index_by_replica()
for kind in ("router engine:<hash>", "router <hash>"):
    print(f"{kind:22s} before restart {census_before[kind]:5d}   after {census_after[kind]:5d}")
print(f"blocks per replica     before restart {index_before}   after {index_after}")
print(f"engine:<hash> keys whose blocks are gone: before restart {orphans_before}, "
      f"after {orphaned_engine_keys()}")
replayed = metric(scrape(EPP_METRICS), "llm_d_epp_kv_cache_index_admissions_total")
print(f"blocks the new router process re-admitted from the replicas' replay buffers: {replayed:.0f}")

after_rows = []
for i, messages in enumerate(restart_conversations):
    messages.append({"role": "user", "content": QUESTIONS[1]})
    after_rows.append({"conversation": i, "turn": 2, **ask(router, messages)})
print()
print_rows(before_rows + after_rows)
follow_up_summary(before_rows + after_rows);

The block keys in database 1 and the blocks per replica are unchanged, and
every follow-up went to the replica that holds its conversation, on the first
request after the restart.

The re-admitted count is worth a look too. When the router connects to a
replica's event stream, it also asks that replica to replay its recent events
(vLLM keeps the last 10,000 engine steps for this, on port 5559). Re-applying
events that are already in the index leaves the block keys and the blocks per
replica unchanged. Only the `engine:<hash>` count can change. Each replica
hashes blocks differently, so when both replicas hold the same block, two
`engine:<hash>` keys point at one block key. When one replica evicts the block,
the router removes that replica's field from the block key. It keeps that
replica's `engine:<hash>` mapping, because the block key is not empty yet, and
after that the mapping is never deleted. The orphan count counts these
leftovers. Replaying the evicting replica's events after a restart can delete
them, but only if the block key is empty at the moment its eviction is
re-applied, and that depends on how the two replicas' replays interleave. So the
`engine:<hash>` count drops by some, all, or none of the orphans, depending
on the run.

In a small demo like this one, the replay covers everything, so an in-memory
index would also be rebuilt within a second of the restart. Valkey helps past that
window. vLLM takes one step per generated token across the whole
batch, so at a few tens of milliseconds per step on a busy server, 10,000 steps
is a few minutes of history. Blocks cached before that are invisible to a
fresh in-memory index, but stay in the Valkey index until vLLM evicts them.

Keeping the index in Valkey has a downside. llm-d-router v1.0.0 can only notice that a
replica restarted while the router is running. It compares each event's
sequence number with the last one it received from that replica, which it holds
in memory, and clears the replica's entries when the number drops
(`pkg/kvevents/zmq_subscriber.go`). A router that starts fresh has nothing to
compare against. If a replica restarts while the router is down, or evicts
blocks while the router is down and those events leave its replay buffer before
the router is back, the new router never removes those entries, and the Valkey
index keeps pointing at blocks the replica no longer has. A fresh in-memory
index would not contain them. In this stack the cost is limited. LMCache still
has that KV in Valkey, so a request routed on a stale entry misses the expected
local hit but loads the prefix from Valkey instead of recomputing it.
Speculative entries have the same problem. The timer that removes them after 2
seconds runs in the router process, so a router that stops inside that window
leaves them in Valkey. That is why the restart cell above waits 2.5 seconds.

Running more than one router needs a shared index, but a shared index is not
enough. llm-d v1.0.0's
[precise prefix-cache routing guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/precise-prefix-cache-routing)
still runs the router as a single replica. The `token-load-scorer` and the
affinity filter use in-flight load, and each router process counts that load
for itself. Two active-active routers would each see half the load, and the
affinity filter's load check would be wrong (see Limits).

## When the best server is busy

Routing to the replica that has the prefix only helps while that replica has
room. The `prefix-cache-affinity-filter` estimates each replica's wait from the
tokens the router has in flight to it, divided by `peakPrefillThroughput`, and
gives up on the sticky replica when its estimated wait is more than
`maxTTFTPenaltyMs` longer than another replica's. This recipe changes three
router settings from llm-d's GPU defaults, all in
[`epp/precise-valkey.yaml`](epp/precise-valkey.yaml):

| Setting | Default | Here | Why |
| --- | --- | --- | --- |
| `inflight-load-producer` `addEstimatedOutputTokens` | `false` | `true` | The default counts only uncached prompt tokens, so a replica busy generating long answers looks idle. With `true`, each request also counts its expected output (its `max_tokens`, capped at 1,000) until it finishes. |
| `prefix-cache-affinity-filter` `peakPrefillThroughput` | `15928` | `1000` | Tokens per second used to turn in-flight tokens into a wait. The default is calibrated for Qwen 32B on two H100 80GB GPUs; the first table above measured this CPU. |
| `prefix-cache-affinity-filter` `maxTTFTPenaltyMs` | `18000` | `1500` | At 1,000 tokens/s, the router leaves the sticky replica once it has 1,500 more in-flight tokens than another replica. |

Each replica runs at most two requests at a time (`--max-num-seqs 2` in
`docker-compose.yml`). The example:

1. Warm a "report template" prefix and a conversation's first turn on replica
   A, directly, so the router's index says A has both, and wait for their KV
   to reach Valkey.
2. Through the router, start two long batch-style requests on the template
   (1,000 output tokens each, `ignore_eos`). Both go to A, which has the
   template; when the second one is routed, A has about 1,000 tokens in flight,
   under the 1,500 limit. A is now full.
3. Send the conversation's next turn through the router. A has about 2,000
   tokens in flight and B none, so the router breaks stickiness and sends the
   conversation to B, which loads it from Valkey.
4. Close the two long requests (vLLM aborts them) and repeat with fresh
   prompts.

The decision depends only on token counts the router itself tracks, so it is
the same on any hardware. The first run also measures the alternatives: the same
conversation sent to A anyway (it waits for a free slot) and a same-sized
conversation that B has to recompute.

In [ ]:
def router_inflight_tokens():
    text = scrape(EPP_METRICS)
    return {name: int(metric(text, "llm_d_epp_inflight_tokens", endpoint_name=name))
            for name in REPLICA_URLS}


def affinity_decisions():
    text = scrape(EPP_METRICS)
    return {outcome: metric(text, "llm_d_epp_prefix_cache_affinity_filter_decisions_total", outcome=outcome)
            for outcome in ("sticky", "load_override", "no_match")}


def start_long_request(prompt, max_tokens=1000):
    """Non-streaming completion through the router; returns the open connection."""
    conn = http.client.HTTPConnection("127.0.0.1", 8080, timeout=600)
    body = {"model": MODEL, "prompt": prompt, "max_tokens": max_tokens,
            "ignore_eos": True, "temperature": 0}
    conn.request("POST", "/v1/completions", json.dumps(body), {"Content-Type": "application/json"})
    return conn


def busy_replica_run(measure_alternatives):
    tag = uuid.uuid4().hex[:8]
    template = f"[report template {tag}] " + document(f"template-{tag}", clauses=26)
    direct["vllm-a"].completions.create(model=MODEL, prompt=template, max_tokens=1)
    conversation = new_conversation("busy")
    conversation.append({"role": "user", "content": QUESTIONS[0]})
    conversation.append({"role": "assistant", "content": ask(direct["vllm-a"], conversation)["text"]})
    conversation.append({"role": "user", "content": QUESTIONS[1]})
    settled_dbsize()  # the conversation's KV is in Valkey before B might need it

    decisions_before = affinity_decisions()
    long_requests = []
    for job in range(2):
        long_requests.append(start_long_request(f"{template}\nReport {job}: write it in full."))
        time.sleep(0.3)  # let the router see the first before routing the second
    time.sleep(0.5)      # demo pacing only: both requests are now decoding on a replica
    inflight = router_inflight_tokens()
    running = {n: int(metric(scrape(f"{u}/metrics"), "vllm:num_requests_running"))
               for n, u in REPLICA_URLS.items()}
    routed = ask(router, conversation)
    alternatives = {}
    if measure_alternatives:
        same_size = new_conversation("busy-cold") + conversation[1:]
        alternatives["B recomputes a same-sized conversation"] = ask(direct["vllm-b"], same_size)
        alternatives["A anyway (waits for a free slot)"] = ask(direct["vllm-a"], conversation)
    for conn in long_requests:  # closing the connection makes vLLM abort the request
        conn.sock.shutdown(socket.SHUT_RDWR)
        conn.close()
    decisions = {k: int(v - decisions_before[k]) for k, v in affinity_decisions().items()}
    return inflight, running, routed, alternatives, decisions


results = []
for run in range(3):
    inflight, running, routed, alternatives, decisions = busy_replica_run(measure_alternatives=(run == 0))
    results.append(routed["replica"])
    print(f"run {run + 1}: before the turn, router in-flight tokens {inflight}, "
          f"vLLM running requests {running}")
    print(f"       affinity filter decisions {decisions}")
    print(f"       conversation -> {routed['replica']}: TTFT {routed['ttft_ms']} ms, "
          f"{routed['valkey']} tokens loaded from Valkey, {routed['local']} local "
          f"(of {routed['prompt']})")
    for label, alt in alternatives.items():
        print(f"       for comparison, {label}: TTFT {alt['ttft_ms']} ms "
              f"({alt['local']} local, {alt['valkey']} from Valkey)")
time.sleep(1)
print(f"\nconversation moved to the idle replica in {results.count('vllm-b')}/3 runs; "
      f"router in-flight tokens after aborting the long requests: {router_inflight_tokens()}")

In each run you should see the two long requests stay on A (`sticky` 2) and the
conversation go to B (`load_override` 1). B has never seen the conversation,
but most of it comes from Valkey, so its first token arrives in a fraction of
the time B needs to recompute a conversation of the same size: on an 8-core
laptop, roughly 0.25 to 0.5 s against about 2.2 s. Sending it to A anyway would
be a local hit, but only after A finishes one of its long jobs, which takes
around 16 s.

This example also shows one of the limits below. The router does not know
that B can load the conversation from Valkey: it scores B as if B had to
recompute every token. Here A was busy enough that B still won, but in a closer
case the router underrates a neighbor that Valkey would make cheap.

## Batch jobs: a queue and per-user quotas in Valkey

llm-d-async is llm-d's batch processor. It pulls jobs from a Valkey sorted set,
checks a per-user quota, sends admitted jobs to an inference endpoint, and
pushes results to a Valkey list. Everything it needs to know is in
[`async/transport-config.json`](async/transport-config.json):

```json
"url": "redis://valkey:6379",
"queues": [{
  "queue_name": "request-sortedset",   "result_queue_name": "result-list",
  "igw_base_url": "http://envoy:8081", "request_path_url": "/v1/completions",
  "gate_type": "redis-quota",
  "gate_params": {"address": "valkey:6379", "attribute": "userid",
                  "mode": "rate-limit", "limit": 3, "window": "20s", "prefix": "quota:"}
}]
```

`igw_base_url` points at Envoy, so batch jobs go through the same router as
interactive traffic and land on whichever replica has their prefix cached.

Jobs normally come from llm-d-async's Go producer library. Here Python submits
them by doing exactly what that producer does: build the same JSON envelope
and run the same Lua script, which increments a per-deadline sequence number,
records the job's active token, and adds the job to the sorted set with a score
of `deadline + seq / 2^21` (earliest deadline first, then submission order).
The script below is copied from llm-d-async v1.0.0
(`producer/redis_sortedset_producer.go`).

In [ ]:
QUEUE, RESULTS = "request-sortedset", "result-list"

# submitRequestScript from llm-d-async v1.0.0 producer/redis_sortedset_producer.go.
SUBMIT_REQUEST = Script("""
local seq = redis.call("INCR", KEYS[1])
redis.call("EXPIREAT", KEYS[1], ARGV[3])
redis.call("DEL", KEYS[2])
redis.call("SET", KEYS[3], ARGV[1], "PX", ARGV[2])
local score = tonumber(ARGV[6]) + math.min(seq, 2097151) / 2097152
redis.call("ZADD", KEYS[4], string.format("%.17g", score), ARGV[4] .. seq .. ARGV[5])
if KEYS[5] then
  redis.call("SET", KEYS[5], ARGV[7], "PX", ARGV[8])
end
return seq
""")


def go_json(value):
    """JSON the way Go's encoding/json writes it: compact, UTF-8, <>& escaped."""
    text = json.dumps(value, separators=(",", ":"), ensure_ascii=False)
    for char, escaped in (("<", "\\u003c"), (">", "\\u003e"), ("&", "\\u0026"),
                          ("\u2028", "\\u2028"), ("\u2029", "\\u2029")):
        text = text.replace(char, escaped)
    return text


def submit_job(job_id, prompt, userid, max_tokens=16, ttl_s=300):
    """Enqueue one /v1/completions job the way llm-d-async's producer does."""
    created = int(time.time())
    deadline = created + ttl_s
    token = secrets.token_hex(16)
    # Field order matches the Go structs (api.InternalRouting, api.RequestMessage).
    envelope = go_json({
        "internal": {"request_token": token, "request_queue_name": QUEUE,
                     "result_queue_name": RESULTS, "enqueue_seq": -1},
        "request_kind": "plain",
        "data": {"id": job_id, "created": created, "deadline": deadline,
                 "payload": {"model": MODEL, "prompt": prompt, "max_tokens": max_tokens,
                             "temperature": 0},
                 "metadata": {"userid": userid}, "endpoint": "/v1/completions",
                 "model": MODEL},
    })
    # The script splices the sequence number in where the -1 placeholder is.
    head, tail = envelope.split('"enqueue_seq":-1', 1)
    keys = [f"request-seq:{QUEUE}:{deadline}", f"request-cancel:{job_id}",
            f"request-active:{job_id}", QUEUE]
    args = [token, str(max(int((deadline - time.time()) * 1000), 1)), str(deadline + 3600),
            head + '"enqueue_seq":', tail, str(deadline)]
    return valkey.invoke_script(SUBMIT_REQUEST, keys=keys, args=args)


def gate_decisions():
    text = scrape(ASYNC_METRICS)
    return {"dispatched": metric(text, "llm_d_async_async_dispatched_requests_total"),
            "quota_exhausted": metric(text, "llm_d_async_async_gate_decisions_total",
                                      reason="quota_exhausted")}


print("submit_job() ready")

One user, `analyst-<id>`, submits five questions about the same document. The
quota allows three jobs per user per 20 seconds, so three run straight away and
two wait in the sorted set until the window moves on. The cell polls Valkey once
a second and prints the queue, the quota key, and the results as they arrive.

In [ ]:
run_id = uuid.uuid4().hex[:8]
user = f"analyst-{run_id}"
quota_key = f"quota:userid:{user}"
batch_document = document(f"batch-{run_id}", clauses=26)
job_ids = [f"job-{run_id}-{i}" for i in range(5)]

counters_before, gates_before = replica_counters(), gate_decisions()
submitted_at = time.monotonic()
for i, job_id in enumerate(job_ids):
    submit_job(job_id, f"{batch_document}\n\nQuestion {i}: summarise clause {i * 5}.\nAnswer:", user)

rows = valkey.custom_command(["ZRANGE", QUEUE, "0", "-1", "WITHSCORES"])
print(f"queued: {len(rows)} jobs in {QUEUE} (sorted set)")
member, score = next(iter(rows.items()))
print(f"one member (score {score}):\n  {member.decode()[:240]}...")

results = {}
print("\n  t   queued  quota count  quota TTL  results")
while len(results) < len(job_ids) and time.monotonic() - submitted_at < 90:
    while (raw := valkey.rpop(RESULTS)) is not None:  # the producer's GetResult pops from the right
        result = json.loads(raw)
        if result["id"] in job_ids:
            results[result["id"]] = result
    print(f"{time.monotonic() - submitted_at:4.0f}s {valkey.zcard(QUEUE):6d} "
          f"{valkey.zcard(quota_key):12d} {valkey.ttl(quota_key):10d} {len(results):8d}")
    time.sleep(1)

counters_after, gates_after = replica_counters(), gate_decisions()
print("\njob     status  cached  answer")
for job_id in job_ids:
    result = results.get(job_id)
    if result is None:
        print(f"{job_id[-1]:>3s}     missing")
        continue
    body = json.loads(result["payload"])
    usage = body.get("usage", {})
    cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens", 0)
    answer = body["choices"][0]["text"].strip().replace("\n", " ")[:50] if "choices" in body else body
    print(f"{job_id[-1]:>3s}  {result.get('status_code', result.get('error_code'))!s:>8s}  {cached:6d}  {answer}")
print("\nserved through the router by:",
      {n: int(counters_after[n]["served"] - counters_before[n]["served"]) for n in REPLICA_URLS})
print("gate decisions:", {k: int(gates_after[k] - gates_before[k]) for k in gates_after})

The first three jobs were admitted at once. The quota is a sorted set per user
(`quota:userid:<user>`) with one member per admitted job, scored by admission
time, and an expiry of twice the window. With three members inside the 20-second
window, the other two jobs stayed in `request-sortedset` until the first
admissions aged out. `quota_exhausted` counts refusals, and one job can be
refused many times. On every poll (every 500 ms here) the processor dequeues
each waiting job, the gate refuses it again, and each refusal counts once
(`async_gate_decisions_total` in llm-d-async v1.0.0 `pkg/metrics/metrics.go`).
Two jobs held for about 20 seconds come to about 2 × 40 refusals, so expect the
counter to end up around 80. Jobs that went to a replica already holding the shared document show it
in the `cached` column: the router routes batch jobs by prefix exactly like chat
requests.

The queue keys are now part of the census alongside the router index and the
LMCache chunks:

In [ ]:
print("batch queue keys:", sorted(k for k in scan_keys(valkey)
                                  if k in QUEUE_KEYS or k.startswith(QUEUE_PREFIXES))[:8], "...")
show_census("Valkey now")

## Limits

- **The routing index connects to a single Valkey node, so it can't use a
  sharded cluster or IAM authentication yet.** In llm-d-router v1.0.0 the index
  (`pkg/kvcache/kvblock/redis.go`) builds one `redis.NewClient` from
  `redis.ParseURL(address)`: one standalone node, with a password or TLS only if
  they fit in the URL, and no cluster client or token-refresh hook. (The batch
  processor is only partly limited. llm-d-async v1.0.0's queue transport
  accepts `mode: cluster` and `mode: sentinel`, but the `redis-quota` gate used
  here takes a single `address` and opens a standalone connection to it
  (`docs/transports.md`). The producer script this notebook copies,
  `submitRequestScript`, is also standalone-only, because its keys span several
  cluster slots. In cluster mode the Go producer enqueues in two steps instead.)
- **The router can't see what LMCache stores in Valkey, so it routes only on
  what each server holds in its own GPU or CPU memory.** The router can score other
  storage tiers (llm-d-router v1.0.0 `pkg/kvcache/backend.go` has weights for
  `cpu`, `storage`, `shared_storage` and `object_store`), but nothing sends it
  events for them: the KV connector vLLM 0.28 loads here, `LMCacheMPConnector`
  (from LMCache 0.5.4), returns no events from `take_events()`. You saw the
  consequence in the busy-server example: the router assumed the neighbor had
  to recompute the whole conversation, when most of it came from Valkey.
- **LMCache's Valkey login bug breaks password-protected Valkey, standalone or
  cluster, until its one-line fix
  ([LMCache/LMCache#5917](https://github.com/LMCache/LMCache/pull/5917)) is
  released.** LMCache 0.5.4's Valkey adapter accepts a `username` and
  `password`, but passes them to valkey-glide in the wrong order
  (`ServerCredentials(username, password)` in
  `lmcache/v1/storage_backend/valkey/worker_pool.py`, where glide expects the
  password first), so the login fails. This stack, like the KV caching recipe,
  runs Valkey without authentication.
- **llm-d runs one router, so several router copies can't share this index
  yet.** llm-d v1.0.0's
  [precise prefix-cache routing guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/precise-prefix-cache-routing)
  runs the router as a single replica, and names two things that running
  several routers side by side would need: shared in-flight load and a complete
  index when a router starts. An index in Valkey gives the second, because a
  router that starts later reads everything the index already holds. It does not
  give the first. The `token-load-scorer` and the affinity filter's load check
  use in-flight tokens that each router process counts for itself, so two
  routers would each see about half the load and move conversations at the
  wrong time. This recipe shows the index surviving a router restart, not two
  routers sharing it.

## How it works

| Component | Role | Valkey use |
| --- | --- | --- |
| Envoy | Front door on `localhost:8080`; asks the EPP where each request goes (ext_proc) and forwards it | none |
| llm-d EPP (Endpoint Picker) v1.0.0 | Tokenizes each request, scores replicas by cached prefix and load, picks one | prefix-cache index (db 1): `<hash>` hashes and `engine:<hash>` sorted sets |
| vLLM 0.28 (x2, CPU) | Serves the model; keeps its own prefix cache; publishes KV events over ZMQ | none directly |
| LMCache 0.5.4 server | Takes KV from both replicas and serves it back | KV chunks (db 0): one string per 256 tokens |
| llm-d-async v1.0.0 | Pulls batch jobs, applies per-user quotas, sends jobs through Envoy | `request-sortedset`, `result-list`, `quota:*`, `request-*` (db 0) |
| Valkey 9.1 (valkey-bundle) | One server for all of the above | |

```text
 chat requests                         batch jobs (submitted from Python)
      |                                       |
      v                                       v
 Envoy :8080 <--ext_proc--> EPP          Valkey db 0: request-sortedset, quota:*
      |                     | ^                |
      |  index in Valkey db 1 | KV events      v
      v                       | (ZMQ)    llm-d-async --> Envoy (same path as chat)
 vllm-a, vllm-b --------------+
      | ^
      v | KV chunks
 LMCache server <--> Valkey db 0: Qwen/...@... chunks
```

## Configuration reference

The settings this recipe sets on purpose. Everything else is the upstream
default.

| File | Setting | Value | Why |
| --- | --- | --- | --- |
| `epp/precise-valkey.yaml` | `redisConfig.address` | `redis://valkey:6379/1` | Router index in Valkey, in a database of its own |
| `epp/precise-valkey.yaml` | `blockSizeTokens` | `64` | Must equal vLLM `--block-size` |
| `epp/precise-valkey.yaml` | `addEstimatedOutputTokens` | `true` | Count decode work as load |
| `epp/precise-valkey.yaml` | `peakPrefillThroughput` | `1000` | CPU prefill rate (tokens/s) |
| `epp/precise-valkey.yaml` | `maxTTFTPenaltyMs` | `1500` | Leave the sticky replica at 1,500 more in-flight tokens |
| `epp/endpoints.yaml` | `address` | `172.16.89.11`, `.12` | Must equal each replica's `POD_IP` and compose address |
| `docker-compose.yml` (vLLM) | `--enable-prefix-caching --block-size 64` | | Local prefix cache the router routes towards |
| `docker-compose.yml` (vLLM) | `--kv-events-config` | ZMQ on 5556, replay on 5559, topic `kv@<POD_IP>:8000@<model>` | Feeds the router's index |
| `docker-compose.yml` (vLLM) | `--kv-cache-memory-bytes 96M` | 128 blocks of 64 tokens | Room for this notebook's conversations on each replica |
| `docker-compose.yml` (vLLM) | `--max-num-seqs 2` | | Two long jobs fill a replica (busy-server example) |
| `docker-compose.yml` (LMCache) | `--l2-adapter` | `{"type":"valkey","startup_nodes":"valkey:6379"}` | KV chunks in Valkey |
| `async/transport-config.json` | `url`, `gate_params.address` | `redis://valkey:6379`, `valkey:6379` | Queue and quota in Valkey |
| `async/transport-config.json` | `igw_base_url` | `http://envoy:8081` | Batch jobs go through the router |
| `async/transport-config.json` | `limit`, `window` | `3`, `20s` | Per-user quota |

## Cleanup

```bash
docker compose down
```

This removes the containers, and with them everything in Valkey (it runs
without persistence). The `hf-cache` volume with the model weights is kept, so
the next `docker compose up` skips the download; `docker compose down -v`
removes it too.

## What you learned

- **One Valkey, three jobs.** The router's prefix-cache index, LMCache's KV
  chunks and llm-d-async's queue and quotas all ran on one Valkey, each set up
  with a URL in a config file. The router index needs a database of its own.
- **Routing and a shared cache solve different problems.** The router sent
  follow-ups back to the replica that had them in memory (local hits); random
  routing sent some elsewhere, where LMCache loaded them from Valkey instead of
  recomputing them.
- **The index outlives the router.** After a restart the router's block keys
  in Valkey were unchanged and routing was correct from the first request.
  The router also replays recent vLLM events on connect; the Valkey index does
  not depend on that replay window. But a router started after a replica
  restarted cannot tell, so stale entries can stay behind.
- **Busy servers spill over cheaply.** When the replica holding a conversation
  was full, the router moved the conversation, and the other replica loaded most
  of it from Valkey.
- **Batch jobs share the same path.** Jobs queued in a Valkey sorted set,
  limited per user by a Valkey quota, went through the same router and benefited
  from the same prefix routing.

## Next steps

- **Add replicas.** Add a `vllm-c` service with the next address, its `POD_IP`
  and a line in `epp/endpoints.yaml`; the router subscribes to its events and the
  index and KV cache are shared with it automatically.
- **Deploy on Kubernetes.** llm-d's
  [precise prefix-cache routing guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/precise-prefix-cache-routing)
  covers the Kubernetes deployment. It runs one router replica: two EPP
  instances pointed at the same Valkey database would share one index, but
  each would track only its own in-flight load.
- **Production batch.** llm-d's
  [batch serving guide](https://github.com/llm-d/llm-d/tree/v1.0.0/guides/batch-serving)
  covers llm-d-async on Kubernetes, and llm-d-async's
  [transport docs](https://github.com/llm-d/llm-d-async/blob/v1.0.0/docs/transports.md)
  describe its Valkey cluster and sentinel modes for the queue (the quota gates
  still connect to one node).
- **Size the KV cache.** The [KV caching recipe](../kv-caching/) covers LMCache's
  Valkey options, including TTLs to bound how much KV Valkey keeps.